# AOCR

## **Progetto - ABGA**

### Dati

$N$   items, $n \in N$

$M$   unità, $m \in M$

$b$ capacità batteria

$q_{m,n}$  risorse richieste

$e_j$ dispendio energetico di un job $j$

$d_j$ tempo di erogazione di un job $j$

$R = \{1, …, n\}$ set di operazioni di ricarica

 $t$  tempo di caricamento batteria (costante)

In [1]:
# INPUT Data
import re

filename = "Ins_V10_J200_T30_R60_B10_W4_S359_N9" + ".txt"

file = "./ASP-BC Instances/" + filename

text = open(file).read()

# lettura tramite regex
M, N, t, b = map(int, re.search(
    r"N_MACHINES:(\d+)\s*N_JOBS:(\d+)\s*CHARGING_TIME:(\d+)\s*INITIAL_CHARGE:(\d+)",
    text
).groups())

# prendo solo la prima colonna
d = [float(x) for x in re.search(r"D:\[(.*?)\]", text, re.S).group(1).split()][::2]
e = [float(x) for x in re.search(r"w:\[(.*?)\]", text, re.S).group(1).split()][::2]



print(f"# Jobs: ",N,"\n# AVG: ", M,"\nCapacità batterie: ",b,"\nTempo di ricarica: ",t)

for j in range(N):
    print(f"Job ",j+1,"; durata: ",d[j],"; costo energetico: ",e[j])


# Jobs:  200 
# AVG:  10 
Capacità batterie:  10 
Tempo di ricarica:  60
Job  1 ; durata:  62.0 ; costo energetico:  7.1
Job  2 ; durata:  62.0 ; costo energetico:  7.1
Job  3 ; durata:  62.0 ; costo energetico:  7.1
Job  4 ; durata:  62.0 ; costo energetico:  7.1
Job  5 ; durata:  62.0 ; costo energetico:  7.1
Job  6 ; durata:  11.0 ; costo energetico:  3.7
Job  7 ; durata:  11.0 ; costo energetico:  3.7
Job  8 ; durata:  11.0 ; costo energetico:  3.7
Job  9 ; durata:  11.0 ; costo energetico:  3.7
Job  10 ; durata:  11.0 ; costo energetico:  3.7
Job  11 ; durata:  52.0 ; costo energetico:  0.9
Job  12 ; durata:  52.0 ; costo energetico:  0.9
Job  13 ; durata:  52.0 ; costo energetico:  0.9
Job  14 ; durata:  52.0 ; costo energetico:  0.9
Job  15 ; durata:  52.0 ; costo energetico:  0.9
Job  16 ; durata:  44.0 ; costo energetico:  4.0
Job  17 ; durata:  44.0 ; costo energetico:  4.0
Job  18 ; durata:  44.0 ; costo energetico:  4.0
Job  19 ; durata:  44.0 ; costo energetico:  4.0
Job  

## Aproccio Esatto


Variante del **Bottleneck Generalized Assignment Problem**

$x_j^m \in \{0,1\}$ : uguale a 1 se il transfer job $j$ viene erogato da $m$

$q_r^m \in \{0,1\}$ : uguale a 1 se il charging job $r$ viene erogato da $m$

$y_{j,r}^m \in \{0,1\}$ : uguale a 1 se il transfer job $j$ dopo il charging job $r$ viene erogato da $m$

$C_{max} \in \Re^+$  : Tempo di completamento del processo di trasferimento

$z = minC_{max}$

$C_{max} \ge \sum_{j\in J}d_j*x_j^m + \sum_{r \in R \backslash \{ 1\}} t*q_r^m  \ \ \  \forall \ m \in M$     :  Il tempo di completamento deve essere maggiore o uguale del tempo di erogazione dei job j eseguiti dalle unità m e il tempo di ricarica t per ogni ricarica eseguita da una unità m

$\sum_m x_j^m = 1 \ \forall \ j \in J$ : Ogni job può essere svolto solo da una unità

$\sum_r y_{j,r}^m = x_j^m  \ \ \forall j,m$   : la somma delle operazioni di ricarica associate ad un job ed una unità è uguale alla variabile binaria di transfer job erogato da tale unità

$2y_{j,r}^m \le x_j^m + q_r^m \ \ \forall j,r,m$ :   $y_{jr}^m$ può valere 1 solo se sia $x_j^m = 1$(il job j è assegnato all'AGV m) sia $q_r^m = 1$ (la ricarica r è assegnata allo stesso AGV m).

$q_r^m \le q_{r-1}^m \ \ \forall \ m,r \in R \backslash \{1 \}$ : symmetry breaking: le ricariche vanno "attivate" in ordine

 $q_1^m = 1 \ \forall \ m$ : ogni unità parte completamente carica

$\sum_{j \in J} e_j\, y_{jr}^m \le b \qquad \forall m \in M,\ r \in R$  : per ogni AGV $m$ e per ogni "blocco di ricarica" $r$, la somma dei consumi energetici $e_j$ di tutti i job eseguiti in quel blocco (cioè con $y_{jr}^m=1$) non può superare la capacità di batteria $b$. Senza questo vincolo il modello non avrebbe alcun legame con la batteria — sarebbe solo uno scheduling puro su macchine parallele.


In [ ]:
import gurobipy as gp

model = gp.Model("ASP_BC")

x = model.addVars(M,N,vtype=gp.GRB.BINARY, name="x")
q = model.addVars(M,N,vtype=gp.GRB.BINARY, name="q")
y = model.addVars(M,N,N,vtype=gp.GRB.BINARY, name="y")

C_max = model.addVar(vtype=gp.GRB.CONTINUOUS, name="C_max", lb=0)

# FUNZIONE OBIETTIVO
z = model.setObjective(C_max, gp.GRB.MINIMIZE)

model.addConstrs(C_max >= (gp.quicksum(d[j]*x[m,j] for j in range(N)) + gp.quicksum(t*q[m,r] for r in range(1,N))) for m in range(M))

model.addConstrs(gp.quicksum(x[m,j] for m in range(M)) == 1 for j in range(N))

model.addConstrs(gp.quicksum(y[m,j,r] for r in range(N)) == x[m,j] for m in range(M) for j in range(N))

model.addConstrs(2*y[m,j,r]<= x[m,j]+q[m,r] for m in range(M) for j in range(N) for r in range(N))

model.addConstrs(q[m,r] <= q[m,r-1] for m in range(M) for r in range(1,N))

model.addConstrs(q[m,0] == 1 for m in range(M))

model.addConstrs(gp.quicksum(e[j]*y[m,j,r] for j in range(N)) <= b for m in range(M) for r in range(N))

In [ ]:
# model.setParam("TimeLimit", 600)   # Limite di 600 secondi / 10 minuti

solution = model.optimize()

In [ ]:
print("Makespan: ", model.objVal)

#x
for (m, j), var in x.items():
    print(f"AGV {m + 1}, Job {j + 1}: {var.X}")

#q
k = 0
for (m, r), var in q.items():
    if var.X == 1:
        k += 1
        print(f"AGV {m + 1}, Charge {r + 1}: {var.X}")

print(f"Total charges used: {k - M}") # Sottraiamo M perché ogni AGV parte con una carica iniziale

# AVG e blocchi di job
for m in range(M):
    print(f"\nAGV {m + 1}")

    for r in range(N):
        jobs = [
            j for j in range(N)
            if y[m, j, r].X > 0.5
        ]

        if not jobs:
            continue

        nomi_job = ", ".join(f"J{j + 1}" for j in jobs)
        energia = sum(e[j] for j in jobs)
        durata = sum(d[j] for j in jobs)

        if r == 0:
            print("  Blocco iniziale:")
        else:
            print(f"  Ricarica (durata {t})")
            print(f"  Blocco {r + 1}:")

        print(f"    Job: {nomi_job}")
        print(f"    Energia: {energia} su {b}")
        print(f"    Durata dei job: {durata}")


### Approcio euristico - Suddivisione del problema

**STEP 1 - BIN-PACKING PROBLEM (BPP) Per stabilire il numero minimo di ricariche da effettuare**

$J, R$ rispettivamente set di job e di ricariche

$\gamma_r \in \{0,1\}$ : uguale a 1 se la r-esima ricarica viene erogata, altrimenti 0

$\chi_{r,j} \in \{0,1\}$ : uguale a 1 se il job j è assegnato alla r operazione di ricarica, altrimenti 0

$\zeta = min \sum_r \gamma_r$ : minimo della somma delle ricariche

$\sum_r \chi_{r,j}=1 \ \forall j$ : ogni job è assegnato ad una sola operazione di ricarica

$\sum_j e_j *\chi_{r,j} \le b*\gamma_r$ : se una ricarica viene usata, la somma dell’energia utilizzata da un transfer job è minore della capacità massima della batteria 

$\gamma_r \le \gamma_{r-1} \ \forall r \in R \backslash \{1\}$ : symmetry breaking: le ricariche vanno "attivate" in ordine

sia $\bar\zeta$ il valore ottimo dell’ultima iterazione eseguita dell’algoritmo, allora $max(0,\bar\zeta-|M|)$ è il minimo richiesto di ricariche da effettuare per tutti i job 

$Lower Bound \ (LB) = max \{\lceil \frac{[max(0,\bar\zeta-|M|)]*t+\sum_jd_j }{|M|}\rceil ,\lceil \frac{max(0,\bar\zeta-|M|)}{|M|}*t\rceil\}$ 

Il numeratore rappresenta il tempo totale minimo richiesta da tutti gli AGV per erogare tutti i trasferimenti ed eseguire le ricariche. Il secondo termine ci permette di considerare l’indivisibilità delle operazioni di ricarica.

Data la natura del problema, ci si aspetterebbe che la soluzione ottima utilizzi il minimo numero di ricariche, ma in realtà non è così.


In [ ]:
import gurobipy as gp
import math

model = gp.Model("ASP_S1")

gamma = model.addVars(N, vtype=gp.GRB.BINARY, name="gamma")
chi = model.addVars(N,N, vtype=gp.GRB.BINARY, name="chi")

model.addConstrs(gp.quicksum(chi[r,j] for r in range(N)) == 1 for j in range(N))

model.addConstrs(gp.quicksum(e[j]*chi[r,j] for j in range(N)) <= b*gamma[r] for r in range(N))

model.addConstrs(gamma[r] <= gamma[r-1] for r in range(1,N))

model.setObjective(gp.quicksum((gamma[r]) for r in range(N)), gp.GRB.MINIMIZE)

#model.setParam("TimeLimit", 300)   # Limite di 5 minuti

model.optimize()

zeta_sol = model.objVal

print(f"\n\n--------Numero minimo di ricariche da effettuare: ",max(zeta_sol - M ,0))

# Lower bound teorico

lb_term1 = math.ceil((max(0, zeta_sol - M) * t + sum(d)) / M)
lb_term2 = math.ceil((max(0, zeta_sol - M) / M) * t)

lb = max(lb_term1,lb_term2)

print(f"----------Lower Bound Teorico Makespan: ", lb)


**STEP 2 - GENERALIZED BOTTLENECK ASSIGNMENT PROBLEM per trovare una soluzione ammissibile**

La soluzione dello scorso problema viene usata come input di questo

in particolare sia  $\bar\zeta$ la funzione obbiettivo ottimale del problema definito in precedenza

se $\bar\zeta \le |M|$  allora esiste una soluzione possibile che non include alcuna operazione di ricarica,

altrimenti, almeno un’operazione di ricarica è necessaria.

Per ogni casistica c’è un metodo per determinare una soluzione ammissibile:

**Soluzione senza operazioni di ricarica**

Questa casistica può verificarsi se il numero di job è relativamente piccolo rispetto agli AGV e/o i consumi sono relativamente bassi rispetto alla capacità della batteria

$z = min C_{max}$

$C_{max} \ge \sum_jd_j*x_j^m$ : Garantisce la coerenza della makespan e la durata dei transfer jobs

$\sum_mx_j^m =1$ : Tutti i transfer job sono stati eseguiti

$\sum_j e_j *x_j^m \le b$ :massimo consumo di energia per ogni AGV

**Soluzione con operazioni di ricarica**

In questo caso, sfruttiamo la soluzione del problema precedente BPP per settare un problema che assume che le operazioni di ricarica contengano le operazioni di trasferimento come oggetti e i veicoli come cestini (problema di tipo BGAP-R)

$(\bar\chi,\bar\gamma)$  Soluzione ottimale.

$\bar R$ il più piccolo subset di charging job in una soluzione ammissibile $(\bar R = \{ r \in R : \gamma_r = 1\})$

Ogni operazione di ricarica $\bar r \in \bar R$ ha una durata $D_{\bar r} = (\sum_j d_j * \bar\chi_{\bar r ,j})+t$ 

$z = min C_{max}$

$\theta_{\bar r,m}$ : variabile binaria, vale 1 se l’operazione di ricarica $\bar r$ è assegnata a *m*

$\sum_m\theta_{\bar r,m}= 1$ : garantisce che ogni operazione di ricarica sia associata ad un veicolo

$C_{max} \ge (\sum_{\bar r}D_{\bar r}*\theta_{\bar r, m}) -t$ : coerenza tra la durata del trasporto e ricarica dei veicoli


In [ ]:
# Soluzione precedente

gamma_sol = {r: var.X for r, var in gamma.items()}
chi_sol = {(r, j): var.X for (r, j), var in chi.items()}

K = max(0, zeta_sol - M)

# Step 2
model = gp.Model("ASP_S2")

C_max = model.addVar(vtype=gp.GRB.CONTINUOUS, name="C_max", lb=0)
model.setObjective(C_max, gp.GRB.MINIMIZE)
model.addConstr(C_max >= lb) # Constraint al LB
if K == 0:
    x = model.addVars(M, N, vtype=gp.GRB.BINARY, name="x")

    model.addConstrs(gp.quicksum(x[m, j] for m in range(M)) == 1 for j in range(N))

    model.addConstrs(C_max >= gp.quicksum(d[j] * x[m, j] for j in range(N)) for m in range(M))

    model.addConstrs(gp.quicksum(e[j] * x[m, j] for j in range(N)) <= b for m in range(M))

else:
    active_blocks = {r for r in range(N) if gamma_sol[r] == 1}

    theta = model.addVars(active_blocks, range(M),vtype=gp.GRB.BINARY, name="theta")


    model.addConstrs(gp.quicksum(theta[r, m] for m in range(M)) == 1 for r in active_blocks)


    D = {r: sum(d[j] * chi_sol[r, j] for j in range(N)) + t for r in active_blocks}

    model.addConstrs(C_max >= gp.quicksum(D[r] * theta[r, m] for r in active_blocks) - t for m in range(M))


# model.setParam("TimeLimit", 180)   # Limite di 3 minuti

model.optimize()

print("Makespan: ", model.objVal)


**STEP 3 - LOCAL SEARCH per migliorare la soluzione trovata**

esplora un intorno di possibili soluzioni usando 3 mosse considerabili come add, swap e remove.

**add:** si crea una **nuova** operazione di ricarica su un altro AGV e vi si sposta un job; utile quando la soluzione ottima richiede **più ricariche** del minimo teorico.

sia $(\hat x, \hat y) , \hat C_{max}$ la soluzione ottenuta alla fine dello step 2 ed il corrispondente valore della funzione obiettivo

$\hat c_m = \sum_jd_j*\hat x_j^m + \sum_{r \in R \backslash \{1\}}t*\hat q_r^m$ : la durata del processo di trasferimento

$\hat J_m = \{ j \in J: \hat x_j^m =1\}$ : l’insieme dei job di trasferimento assegnati al veicolo m

$s_a(m_1,j,m_2)$ : risparmio ottenuto aggiungendo un job j da m1 a una nuova operazione di ricarica m2

$s_a(m_1,j,m_2) = max \{0,\hat C_{max}-max(\hat c_{m1} - d_j, \hat c_{m2}+d_j+t,\hat c_m)_{m \in M \backslash \{m1,m2\}}\}$

Si precisa che, se un job $j^*$è l’**unico** assegnato a un’operazione di ricarica $r^*$ dell’AGV $m^*_1$, allora la nuova durata del processo di trasferimento sull’AGV $m^*_1$ diventa $\hat{c}_{m^*_1} - d_j - t$.

**remove:** si sposta un job su una ricarica **già esistente** di un altro AGV, rispettando il vincolo di batteria; serve a sfruttare meglio le ricariche già pianificate.

$\hat R_m = \{r \in R: \hat q_r^m = 1\}$ : l’insieme delle operazioni di ricarica assegnate al veicolo m nella soluzione corrente

$\hat b_r^m = \sum_je_j \hat y_{jr}^m$ : energia richiesta dai job di trasferimento assegnati all’operazione di ricarica r del veicolo m

$s_r(m_1,j,m_2,r)$ : risparmio ottenuto rimuovendo un job j dal veicolo m1 e assegnandolo all’operazione di ricarica r del veicolo m2 senza superare la capacità della batteria $b-\hat b_r^m \ge e_j$

$s_r(m_1,j,m_2,r) =  max \{0,\hat C_{max}-max(\hat c_{m1} - d_j, \hat c_{m2}+d_j,\hat c_m)_{m \in M \backslash \{m1,m2\}}\}$

anche in questo caso, se un job $j^*$è l’**unico** assegnato a un’operazione di ricarica $r^*$ dell’AGV $m_1$ allora la nuova durata del processo di trasferimento sull’AGV $m^*_1$ diventa $\hat{c}_{m^*_1} - d_j - t$.

**swap:** si scambiano due job tra due AGV (e due operazioni di ricarica), sempre rispettando i vincoli di batteria; serve a riequilibrare durate e carichi senza aggiungere ricariche.

$s_r(m_1,r_1,j_1,m_2,r_2,j_2)$ : risparmio ottenuto scambiando un job j1 dall’operazione di ricarica r1 del veicolo m1 con un job j2 dall’operazione di ricarica r2 del veicolo m2, senza superare la capacità della batteria $b−b^{r_1}_{m_1}≥e_{j_2}−e_{j_1} \ e \ b−b^{r_2}_{m_2}≥e_{j_1}−e_{j_2}$

$s_r(m_1,r_1,j_1,m_2,r_2,j_2) = max \{0,\hat C_{max}-max(\hat c_{m1} - d_{j_1} + d_{j_2}, \hat c_{m2}+d_{j_1}-d_{j_2},\hat c_m)_{m \in M \backslash \{m1,m2\}}\}$


In [ ]:
if K == 0:
    x_sol = {(m, j): var.X for (m, j), var in x.items()}
else:
    theta_sol = {(r, m): var.X for (r, m), var in theta.items()}

    #ricostruzione delle soluzioni x, q e y a partire da theta_sol e chi_sol
    x_sol = {}
    q_sol = {}
    y_sol = {}

    for m in range(M):
        for j in range(N):
            x_sol[m, j] = 0 # inizializza a 0
            q_sol[m, j] = 0 # inizializza a 0
            for r in range(N): y_sol[m,j,r] = 0 # inizializza a 0

        for r in active_blocks:
            if theta_sol[r,m] > 0.5:
                q_sol[m, r] = 1 # inverso di theta

            for j in range(N):
                y_sol[m, j, r] = 0 # inizializza a 0

                if chi_sol[r, j] > 0.5 and theta_sol[r, m] > 0.5: # ricostruzione della soluzione
                    x_sol[m, j] = 1
                    y_sol[m, j, r] = 1

C_max_sol = model.objVal

def c(m): # durata del trasferimento di m
    return sum(d[j] * x_sol[m, j] for j in range(N)) + t *sum(q_sol[m, r] for r in range(1,N))

def J(m):
    return [j for j in range(N) if x_sol[m, j] == 1]

def add_charge(m1, j, m2):
    global C_max_sol

    for r in active_blocks:
        if y_sol[m1, j, r] == 1:
            if M > 2 :
                c_m = max(c(m) for m in range(M) if m != m1 and m != m2)
            else : c_m = 0
                    
            t_m1 = c(m1) - d[j] - t
            svuota_blocco = True
            for j_ in range(N):
                if y_sol[m1, j_, r] == 1 and j_ != j:
                    t_m1 = c(m1) - d[j]
                    svuota_blocco = False

            t_m2 = c(m2) + d[j]
            blocco2_vuoto = True
            for r_ in active_blocks:
                if q_sol[m2,r_] == 1:
                    t_m2 = c(m2) + d[j] + t
                    blocco2_vuoto = False


            s = C_max_sol - max(t_m1, t_m2, c_m)

            if  s > 0 : # mossa vantaggiosa -> applico
                if blocco2_vuoto:
                    r_new = 0
                else: 
                    r_new = max(r_ for r_ in active_blocks if q_sol[m2,r_] == 1) + 1
                x_sol[m1, j] = 0
                x_sol[m2, j] = 1
                y_sol[m1, j, r] = 0
                y_sol[m2, j, r_new] = 1
                if svuota_blocco:
                    q_sol[m1, r] = 0
                    active_blocks.remove(r)
                q_sol[m2, r_new] = 1
                active_blocks.add(r_new)
                
                C_max_sol -= s
                return s
            else:
                return 0
    return 0

def R(m):
    return [r for r in range(N) if q_sol[m, r] == 1]

def energia_blocco(r,m): # b_r,m
    return sum(e[j]*y_sol[m, j, r] for j in range(N)) # energia richiesta dai job di trasferimento assegnati all’operazione di ricarica r del veicolo m

def remove_job(m1, j, m2, r):
    global C_max_sol

    if q_sol[m2,r] == 1 and b - energia_blocco(r, m2) >= e[j] : # b - b_r,m >= m2
            if M > 2 :
                c_m = max(c(m) for m in range(M) if m != m1 and m != m2)
            else : c_m = 0

            t_m1 = c(m1) - d[j] - t
            svuota_blocco = True
            for j_ in range(N):
                if y_sol[m1, j_, r] == 1 and j_ != j:
                    t_m1 = c(m1) - d[j]
                    svuota_blocco = False

            t_m2 = c(m2) + d[j]

            s = C_max_sol - max(t_m1, t_m2, c_m)

            if  s > 0: # mossa vantaggiosa -> applico
                x_sol[m1, j] = 0
                x_sol[m2, j] = 1
                y_sol[m1, j, r] = 0
                y_sol[m2, j, r] = 1
                if svuota_blocco:
                    q_sol[m1, r] = 0
                    active_blocks.remove(r)
                C_max_sol -= s
                return s
            else:
                return 0

    else: return 0


def swap_jobs(m1,r1,j1,m2,r2,j2):
    global C_max_sol

    if b - energia_blocco(r1,m1) >= e[j2] - e[j1] and b - energia_blocco(r2,m2) >= e[j1] - e[j2] :
            if M > 2 :
                c_m = max(c(m) for m in range(M) if m != m1 and m != m2)
            else : c_m = 0
            
            t_m1 = c(m1) - d[j1] + d[j2]
            
            t_m2 = c(m2) + d[j1] - d[j2]

            s = C_max_sol - max(t_m1, t_m2, c_m)

            if s > 0: # mossa vantaggiosa -> applico
                x_sol[m1,j1] = 0
                x_sol[m1,j2] = 1
                x_sol[m2,j2] = 0
                x_sol[m2,j1] = 1
                y_sol[m1,j1,r1] = 0
                y_sol[m1,j2,r1] = 1
                y_sol[m2,j2,r2] = 0
                y_sol[m2,j1,r2] = 1
                C_max_sol -= s
                return s

            else: return 0
    else: return 0


**Dettagli di implementazione**

Sia S la soluzione ottenuta alla fine del secondo step. La ricerca locale computa il massimo tra tre tipi di risparmio precedentemente descritti per ogni iterazione. 

Inizialmente poniamo il risparmio massimo $s^*=0$ , e computiamo add, remove e swap in ordine. ogni volta che si ottiene un risparmio maggiore di quello attuale, ne aggiorniamo il valore.

Finché $s^* > 0$ significa che è ancora possibile ridurre la makespan. Se $s^* \le 0$ allora ci si ferma.

![image.png](image.png)

In [ ]:
while True:
    saving = 0

    m_max = max(range(M),key=c) # individuo l'AVG con il massimo makespan

    # --- ADD ---
    for j in range(N):
        for m2 in range(M):
            if x_sol[m_max,j] == 1 and m2 != m_max:

                saving_add = add_charge(m_max, j, m2)
                if saving_add > saving:
                    saving = saving_add
                    print(f"Mossa ADD applicata: AGV1 {m_max + 1}; AGV2 {m2 + 1}; Job {j + 1}")

    m_max = max(range(M),key=c) # ricalcolo l'AVG con massimo makespan 
    # --- REMOVE ---
    for (m1,j, r), var in list(y_sol.items()):
        if m1 != m_max or var == 0 : continue
        for m2 in range(M):
            if var == 1 and m2 != m_max:

                saving_remove = remove_job(m_max, j, m2, r)
                if saving_remove > saving:
                    saving = saving_remove
                    print(f"Mossa REMOVE applicata: AGV1 {m_max + 1}; AGV2 {m2 + 1}; Job {j + 1}; ricarica {r + 1}")

    m_max = max(range(M),key=c) # ricalcolo l'AVG con massimo makespan
    # --- SWAP ---
    for (m1, j1, r1), var1 in list(y_sol.items()):
        if m1 != m_max or var1 == 0: continue
        for (m2, j2, r2), var2 in list(y_sol.items()):
            if var1 == 1 and var2 == 1 and m2 != m1 and r2 != r1:
                saving_swap = swap_jobs(m_max, r1, j1, m2, r2, j2)
                if saving_swap > saving:
                    saving = saving_swap
                    print(f"Mossa SWAP applicata: AGV1 {m1 + 1}; ricarica1 {r1 + 1}; Job1 {j1 + 1} "
                          f"-> AGV2 {m2 + 1}; ricarica2 {r2 + 1}; Job2 {j2 + 1}")

    # esci se nessuna mossa ha migliorato o raggiungi il LB

    if saving <= 0 or C_max <= lb:
        break

# Stampa soluzione
print(f"----SOLUZIONE-----")
print(f"Makespan: {C_max_sol}")
print(f"Risparmio dalla soluzione precedente {model.objVal - C_max_sol}")


#x
for (m, j), var in x_sol.items():
    print(f"AGV {m + 1}, Job {j + 1}: {var}")

#q
k = 0
for (m, r), var in q_sol.items():
    if q_sol[m,r] == 1:
        k += 1
        print(f"AGV {m + 1}, Charge {r + 1}: {var}")

print(f"Totale ricariche effettuate: {k - M}") # Sottraiamo M perché ogni AGV parte con una carica iniziale

# AVG e blocchi di job
for m in range(M):
    print(f"\nAGV {m + 1}")

    for r in range(N):
        jobs = [
            j for j in range(N)
            if y_sol[m, j, r] > 0
        ]

        if not jobs:
            continue

        nomi_job = ", ".join(f"J{j + 1}" for j in jobs)
        energia = sum(e[j] for j in jobs)
        durata = sum(d[j] for j in jobs)

        if r == 0:
            print("  Blocco iniziale:")
        else:
            print(f"  Ricarica (durata {t})")
            print(f"  Blocco {r + 1}:")

        print(f"    Job: {nomi_job}")
        print(f"    Energia: {energia} su {b}")
        print(f"    Durata dei job: {durata}")

## Variazione: BPP limitato + Euristica Greedy

Nei problemi di grandi dimensioni, risolvere in modo esatto il BPP può essere molto dispendioso e costituisce un bottleneck di performance per il problema.

Per migliorare questo aspetto, ho pensato di modificare lo step 1 in questo modo:

**BPP con limite temporale**: il modello non troverà necessariamente una soluzione ottima, ma restituirà comunque un **incumbent** $\bar\zeta$, ovvero la migliore soluzione ammissibile trovata, e un **Dual Bound** $\underline\zeta$, ovvero il miglior lower bound dimostrato da Gurobi. Entrambi i valori restano validi anche se l’ottimo non viene raggiunto entro il limite di tempo.

Il dual bound consente di valutare la qualità della soluzione corrente attraverso il gap:

$\text{gap} = \frac{\bar\zeta - \underline\zeta}{\bar\zeta} \times 100$

Se il gap è nullo, la soluzione è già ottima e non è necessario proseguire con la fase euristica; se è ampio, la fase greedy può cercare di migliorare l’incumbent riducendo ulteriormente il numero di blocchi di ricarica.

In [ ]:
import gurobipy as gp
import math

model = gp.Model("ASP_S1")

gamma = model.addVars(N, vtype=gp.GRB.BINARY, name="gamma")
chi = model.addVars(N,N, vtype=gp.GRB.BINARY, name="chi")

model.addConstrs(gp.quicksum(chi[r,j] for r in range(N)) == 1 for j in range(N))

model.addConstrs(gp.quicksum(e[j]*chi[r,j] for j in range(N)) <= b*gamma[r] for r in range(N))

model.addConstrs(gamma[r] <= gamma[r-1] for r in range(1,N))

model.setObjective(gp.quicksum((gamma[r]) for r in range(N)), gp.GRB.MINIMIZE)

model.setParam("TimeLimit", 60)   # Limite di 5 minuti

model.optimize()

# double bound
 
zeta_sol = model.objVal

zeta_lb = model.objBound

print(f"\n\n--------Numero minimo di ricariche da effettuare: ",max(zeta_sol - M ,0))
print(f"--------Dual bound:", zeta_lb - M)
gap = ((zeta_sol - zeta_lb)/zeta_sol) * 100
print(f"--------Gap = ",gap,"%")


# Lower bound teorico

lb_term1 = math.ceil((max(0, zeta_lb - M) * t + sum(d)) / M)
lb_term2 = math.ceil((max(0, zeta_lb - M) / M) * t)

lb = max(lb_term1,lb_term2)

print(f"----------Lower Bound Teorico Makespan: ", lb)


Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2877851
Academic license 2877851 - for non-commercial use only - registered to gi___@studenti.unina.it
Set parameter TimeLimit to value 60
Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (linux64 - "Debian GNU/Linux 13 (trixie)")

CPU model: Intel(R) Core(TM) i7-8700 CPU @ 3.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 6 physical cores, 12 logical processors, using up to 12 threads

Non-default parameters:
TimeLimit  60

Academic license 2877851 - for non-commercial use only - registered to gi___@studenti.unina.it
Optimize a model with 599 rows, 40200 columns and 80598 nonzeros (Min)
Model fingerprint: 0xe494de2f
Model has 200 linear objective coefficients
Variable types: 0 continuous, 40200 integer (40200 binary)
Coefficient statistics:
  Matrix range     [1e-01, 1e+01]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+00]

Found heuristic solutio

**Approccio Greedy**

Cerchiamo di rimuovere il blocco di ricarica con più carica residua, ovvero quello con il minore consumo energetico totale, e cerchiamo di redistribuire i job di quel blocco negli altri blocchi attivi. In caso non si riesca a distribuire, fallback sul secondo blocco con residuo più alto e così via.

Questa è una fase molto aggressiva, che dà un grande vantaggio in caso di riuscita, ma ha anche alte probabilità di fallire.

**Dettagli Ciclo di ricerca greedy**

Il ciclo valuta in ordine di energia residua decrescente i blocchi attivi:

$\bar E_{\bar r} = b - \sum_j \bar\chi_{\bar r,j} \cdot e_j$

dove \(\bar E_{\bar r}\) è l’energia residua del blocco \(\bar r\).

La mossa **remove block** controlla in ordine i job più dispendiosi del blocco target, poiché sono quelli più difficili da piazzare, e cerca tra i blocchi attivi una lista di candidati.

$\bar E_{r^*} \ge e_j \qquad \forall r^* \in \bar R - \{\bar r\}$

è la condizione che un blocco deve soddisfare per essere candidato come destinazione del job \(j\).

Una volta trovati i candidati, si inserisce (fittiziamente) il job nel blocco candidato con **energia residua minima sufficiente** (regola *Best Fit*), così da preservare la capacità dei blocchi più liberi per i job successivi, e si aggiorna l’energia residua del candidato:

$\bar E_{r^*} \leftarrow \bar E_{r^*} - e_j$

I job vengono inseriti definitivamente nei vari blocchi candidati solo se si è sicuri di riuscire a svuotare il blocco in esame, ovvero solo se tutti i job trovano una collocazione valida.

Se per un job non si trovano candidati, si esce dal ciclo senza modificare la soluzione corrente, e si passa al blocco target successivo in ordine di residua decrescente.

Si procede così per tutti i blocchi restanti.

Se il ciclo ha avuto successo (è stato rimosso almeno un blocco di ricarica) allora si procede con una nuova iterazione; altrimenti, se il ciclo fallisce (non si riesce a rimuovere nessun blocco, o si è raggiunto il lower bound), si esce: la soluzione corrente è un minimo locale rispetto alla mossa *remove block*.

In [ ]:
# Soluzione precedente

gamma_sol = {r: var.X for r, var in gamma.items()}
chi_sol = {(r, j): var.X for (r, j), var in chi.items()}
active_blocks = {r for r in range(N) if gamma_sol[r] > 0.5}

def e_residua(r):
    return b - sum(chi_sol[r,j]*e[j] for j in range(N))


def remove_block(r):
    jobs_in_r = [j for j in range(N) if chi_sol[r, j] > 0.5]
    # job più grandi prima
    jobs_in_r.sort(key=lambda j: e[j], reverse=True)

    residua = {r1: e_residua(r1) for r1 in active_blocks if r1 != r}

    move_job, move_to_block = {}, {}
    i = 0
    for j in jobs_in_r:
        candidati = [r1 for r1 in residua if residua[r1] >= e[j]]
        if not candidati:
            return zeta_sol  # fallito: nessun blocco lo accoglie
        # Best Fit: destinazione con residua minima sufficiente
        r1 = min(candidati, key=lambda r1: residua[r1])
        move_job[i] = j
        move_to_block[i] = r1
        residua[r1] -= e[j] # aggiorno l'energia residua r1
        i += 1

    # tutti i job sono stati assegnati: applica la mossa
    for k in range(i):
        chi_sol[r, move_job[k]] = 0
        chi_sol[move_to_block[k], move_job[k]] = 1
    return zeta_sol - 1

# ciclo su tutti i blocchi in ordine di e_residua

while True:
    # if zeta_sol <= zeta_lb : break
    zeta = zeta_sol
    en_blocchi = {r: e_residua(r) for r in active_blocks}

    # ordina per en residua decrescente
    en_blocchi = sorted(en_blocchi.items(), key=lambda item: item[1], reverse=True)

    for r, res in en_blocchi:
        zeta_new = remove_block(r)
        if zeta_new == zeta_sol:
            continue  # rimozione fallita, prova il prossimo
        active_blocks.remove(r)
        zeta_sol = zeta_new

    if zeta == zeta_sol:
        break  # nessun miglioramento in questo giro: minimo locale

# aggiorno gamma
for r in range(N):
    if r in active_blocks : gamma_sol[r] = 1
    else: gamma_sol[r] = 0

print(f"Nuovo numero minimo di ricariche: ", zeta_sol - M)

**Gli step 2 e 3, rimangono invariati**

In [ ]:
K = max(0, zeta_sol - M)

# Step 2
model = gp.Model("ASP_S2")

C_max = model.addVar(vtype=gp.GRB.CONTINUOUS, name="C_max", lb=0)
model.setObjective(C_max, gp.GRB.MINIMIZE)
model.addConstr(C_max >= lb) # Constraint al LB
if K == 0:
    x = model.addVars(M, N, vtype=gp.GRB.BINARY, name="x")

    model.addConstrs(gp.quicksum(x[m, j] for m in range(M)) == 1 for j in range(N))

    model.addConstrs(C_max >= gp.quicksum(d[j] * x[m, j] for j in range(N)) for m in range(M))

    model.addConstrs(gp.quicksum(e[j] * x[m, j] for j in range(N)) <= b for m in range(M))

else:

    theta = model.addVars(active_blocks, range(M),vtype=gp.GRB.BINARY, name="theta")


    model.addConstrs(gp.quicksum(theta[r, m] for m in range(M)) == 1 for r in active_blocks)


    D = {r: sum(d[j] * chi_sol[r, j] for j in range(N)) + t for r in active_blocks}

    model.addConstrs(C_max >= gp.quicksum(D[r] * theta[r, m] for r in active_blocks) - t for m in range(M))


# model.setParam("TimeLimit", 180)   # Limite di 3 minuti

model.optimize()

print("Makespan: ", model.objVal)

In [ ]:
if K == 0:
    x_sol = {(m, j): var.X for (m, j), var in x.items()}
else:
    theta_sol = {(r, m): var.X for (r, m), var in theta.items()}

    #ricostruzione delle soluzioni x, q e y a partire da theta_sol e chi_sol
    x_sol = {}
    q_sol = {}
    y_sol = {}

    for m in range(M):
        for j in range(N):
            x_sol[m, j] = 0 # inizializza a 0
            q_sol[m, j] = 0 # inizializza a 0
            for r in range(N): y_sol[m,j,r] = 0 # inizializza a 0

        for r in active_blocks:
            if theta_sol[r,m] > 0.5:
                q_sol[m, r] = 1 # inverso di theta

            for j in range(N):
                y_sol[m, j, r] = 0 # inizializza a 0

                if chi_sol[r, j] > 0.5 and theta_sol[r, m] > 0.5: # ricostruzione della soluzione
                    x_sol[m, j] = 1
                    y_sol[m, j, r] = 1

C_max_sol = model.objVal

def c(m): # durata del trasferimento di m
    return sum(d[j] * x_sol[m, j] for j in range(N)) + t *sum(q_sol[m, r] for r in range(1,N))

def J(m):
    return [j for j in range(N) if x_sol[m, j] == 1]

def add_charge(m1, j, m2):
    global C_max_sol

    for r in active_blocks:
        if y_sol[m1, j, r] == 1:
            if M > 2 :
                c_m = max(c(m) for m in range(M) if m != m1 and m != m2)
            else : c_m = 0
                    
            t_m1 = c(m1) - d[j] - t
            svuota_blocco = True
            for j_ in range(N):
                if y_sol[m1, j_, r] == 1 and j_ != j:
                    t_m1 = c(m1) - d[j]
                    svuota_blocco = False

            t_m2 = c(m2) + d[j]
            blocco2_vuoto = True
            for r_ in active_blocks:
                if q_sol[m2,r_] == 1:
                    t_m2 = c(m2) + d[j] + t
                    blocco2_vuoto = False


            s = C_max_sol - max(t_m1, t_m2, c_m)

            if  s > 0 : # mossa vantaggiosa -> applico
                if blocco2_vuoto:
                    r_new = 0
                else: 
                    r_new = max(r_ for r_ in active_blocks if q_sol[m2,r_] == 1) + 1
                x_sol[m1, j] = 0
                x_sol[m2, j] = 1
                y_sol[m1, j, r] = 0
                y_sol[m2, j, r_new] = 1
                if svuota_blocco:
                    q_sol[m1, r] = 0
                    active_blocks.remove(r)
                q_sol[m2, r_new] = 1
                active_blocks.add(r_new)
                
                C_max_sol -= s
                return s
            else:
                return 0
    return 0

def R(m):
    return [r for r in range(N) if q_sol[m, r] == 1]

def energia_blocco(r,m): # b_r,m
    return sum(e[j]*y_sol[m, j, r] for j in range(N)) # energia richiesta dai job di trasferimento assegnati all’operazione di ricarica r del veicolo m

def remove_job(m1, j, m2, r):
    global C_max_sol

    if q_sol[m2,r] == 1 and b - energia_blocco(r, m2) >= e[j] : # b - b_r,m >= m2
            if M > 2 :
                c_m = max(c(m) for m in range(M) if m != m1 and m != m2)
            else : c_m = 0

            t_m1 = c(m1) - d[j] - t
            svuota_blocco = True
            for j_ in range(N):
                if y_sol[m1, j_, r] == 1 and j_ != j:
                    t_m1 = c(m1) - d[j]
                    svuota_blocco = False

            t_m2 = c(m2) + d[j]

            s = C_max_sol - max(t_m1, t_m2, c_m)

            if  s > 0: # mossa vantaggiosa -> applico
                x_sol[m1, j] = 0
                x_sol[m2, j] = 1
                y_sol[m1, j, r] = 0
                y_sol[m2, j, r] = 1
                if svuota_blocco:
                    q_sol[m1, r] = 0
                    active_blocks.remove(r)
                C_max_sol -= s
                return s
            else:
                return 0

    else: return 0


def swap_jobs(m1,r1,j1,m2,r2,j2):
    global C_max_sol

    if b - energia_blocco(r1,m1) >= e[j2] - e[j1] and b - energia_blocco(r2,m2) >= e[j1] - e[j2] :
            if M > 2 :
                c_m = max(c(m) for m in range(M) if m != m1 and m != m2)
            else : c_m = 0
            
            t_m1 = c(m1) - d[j1] + d[j2]
            
            t_m2 = c(m2) + d[j1] - d[j2]

            s = C_max_sol - max(t_m1, t_m2, c_m)

            if s > 0: # mossa vantaggiosa -> applico
                x_sol[m1,j1] = 0
                x_sol[m1,j2] = 1
                x_sol[m2,j2] = 0
                x_sol[m2,j1] = 1
                y_sol[m1,j1,r1] = 0
                y_sol[m1,j2,r1] = 1
                y_sol[m2,j2,r2] = 0
                y_sol[m2,j1,r2] = 1
                C_max_sol -= s
                return s

            else: return 0
    else: return 0

In [ ]:
while True:
    saving = 0

    m_max = max(range(M),key=c) # individuo l'AVG con il massimo makespan

    # --- ADD ---
    for j in range(N):
        for m2 in range(M):
            if x_sol[m_max,j] == 1 and m2 != m_max:

                saving_add = add_charge(m_max, j, m2)
                if saving_add > saving:
                    saving = saving_add
                    print(f"Mossa ADD applicata: AGV1 {m_max + 1}; AGV2 {m2 + 1}; Job {j + 1}")

    m_max = max(range(M),key=c) # ricalcolo l'AVG con massimo makespan 
    # --- REMOVE ---
    for (m1,j, r), var in list(y_sol.items()):
        if m1 != m_max or var == 0 : continue
        for m2 in range(M):
            if var == 1 and m2 != m_max:

                saving_remove = remove_job(m_max, j, m2, r)
                if saving_remove > saving:
                    saving = saving_remove
                    print(f"Mossa REMOVE applicata: AGV1 {m_max + 1}; AGV2 {m2 + 1}; Job {j + 1}; ricarica {r + 1}")

    m_max = max(range(M),key=c) # ricalcolo l'AVG con massimo makespan
    # --- SWAP ---
    for (m1, j1, r1), var1 in list(y_sol.items()):
        if m1 != m_max or var1 == 0: continue
        for (m2, j2, r2), var2 in list(y_sol.items()):
            if var1 == 1 and var2 == 1 and m2 != m1 and r2 != r1:
                saving_swap = swap_jobs(m_max, r1, j1, m2, r2, j2)
                if saving_swap > saving:
                    saving = saving_swap
                    print(f"Mossa SWAP applicata: AGV1 {m1 + 1}; ricarica1 {r1 + 1}; Job1 {j1 + 1} "
                          f"-> AGV2 {m2 + 1}; ricarica2 {r2 + 1}; Job2 {j2 + 1}")

    # esci se nessuna mossa ha migliorato o raggiungi il LB

    if saving <= 0:
        break

# Stampa soluzione
print(f"----SOLUZIONE-----")
print(f"Makespan: {C_max_sol}")
print(f"Risparmio dalla soluzione precedente {model.objVal - C_max_sol}")


#x
for (m, j), var in x_sol.items():
    print(f"AGV {m + 1}, Job {j + 1}: {var}")

#q
k = 0
for (m, r), var in q_sol.items():
    if q_sol[m,r] == 1:
        k += 1
        print(f"AGV {m + 1}, Charge {r + 1}: {var}")

print(f"Totale ricariche effettuate: {k - M}") # Sottraiamo M perché ogni AGV parte con una carica iniziale

# AVG e blocchi di job
for m in range(M):
    print(f"\nAGV {m + 1}")

    for r in range(N):
        jobs = [
            j for j in range(N)
            if y_sol[m, j, r] > 0
        ]

        if not jobs:
            continue

        nomi_job = ", ".join(f"J{j + 1}" for j in jobs)
        energia = sum(e[j] for j in jobs)
        durata = sum(d[j] for j in jobs)

        if r == 0:
            print("  Blocco iniziale:")
        else:
            print(f"  Ricarica (durata {t})")
            print(f"  Blocco {r + 1}:")

        print(f"    Job: {nomi_job}")
        print(f"    Energia: {energia} su {b}")
        print(f"    Durata dei job: {durata}")